# 08 · Race simulator (dry races)

Simulates a race thousands of times with any field you choose: drivers from any season they raced (2018–2025), in any team-season's car, at any circuit.

**Each simulated race**
1. Draws one sample from the driver/car model (notebook 07), so uncertainty about skill and cars carries into the results.
2. **Qualifying:** one-lap pace = driver skill + car pace at this circuit, plus qualifying noise. The order sets the grid.
3. **Retirements:** each car retires at its team-season's DNF rate, adjusted for the circuit's DNF rate (dry races; shrunk toward the average when data is thin).
4. **Race:** starting from the grid, a car passes the car ahead only if it's faster by more than the circuit's **overtaking threshold**. Thresholds are calibrated so simulated position changes match each circuit's real dry races.
5. **Points:** 25-18-15-12-10-8-6-4-2-1 (no fastest-lap point, as in 2025).

**Caveat:** for real driver–car pairings, skill + car is well measured. For **swapped** entries (a driver in a car they never drove), results depend on how the model split skill from car, which teammate comparisons only partly pin down; the intervals don't capture all of that. Swapped entries are flagged.

Run notebooks 03, 05 (with circuit information) and 07 first.

In [1]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Mounted at /content/drive
Cloning into '/content/f1-race-sim'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 174 (delta 101), reused 102 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (174/174), 136.81 KiB | 2.58 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [2]:
# @title Install PyMC and the file format for saved fits (skipped if already installed)
import importlib.util
missing = [p for p in ("pymc", "h5netcdf", "h5py") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [3]:
# @title Load the fit, DNF rates and overtaking thresholds
RECALIBRATE = False  # @param {type:"boolean"}

import pandas as pd
import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import clean_dry_laps
from f1sim import skill_model as sm
from f1sim import simulator as sim

paths = get_paths()
results = load_results(paths)
cal = sim.calendar(results)     # fails with instructions if circuit info is missing
idata, data = sm.load_fit(paths)
post = sim.load_posterior(idata, data)

# Dry races = races the model used (no wet tyres, at least some clean laps)
dry_keys = set(map(tuple, clean_dry_laps(load_race_ti("laps", paths))[["Year", "Round"]]
                   .drop_duplicates().to_numpy()))
dnf = sim.dnf_rates(results, dry_keys)

thr_file = paths.models_dir / "overtaking_thresholds.parquet"
if RECALIBRATE or not thr_file.exists():
    print("Calibrating overtaking thresholds (a few seconds)...")
    thresholds = sim.calibrate_overtaking(results, post, dry_keys)
    thresholds.to_parquet(thr_file, index=False)
else:
    thresholds = pd.read_parquet(thr_file)
print(f"{post.n_draws} posterior draws | {len(dry_keys)} dry races | "
      f"overall DNF rate {100 * dnf.overall:.1f}%")
thresholds

Calibrating overtaking thresholds (a few seconds)...
2000 posterior draws | 152 dry races | overall DNF rate 14.6%


,CircuitId,CircuitName,DryRaces,ObservedPosChange,RawThreshold,Threshold,SimPosChange,RawAtLimit
0,monaco,Circuit de Monaco,5,1.52,0.323,0.265,1.73,False
1,baku,Baku City Circuit,7,1.95,0.208,0.189,2.02,False
2,suzuka,Suzuka Circuit,5,1.64,0.209,0.184,1.74,False
3,spa,Circuit de Spa-Francorchamps,6,2.33,0.200,0.180,2.40,False
4,villeneuve,Circuit Gilles Villeneuve,5,2.41,0.203,0.179,2.50,False
5,catalunya,Circuit de Barcelona-Catalunya,8,1.81,0.179,0.167,1.86,False
6,hungaroring,Hungaroring,6,2.55,0.171,0.159,2.60,False
7,mugello,Autodromo Internazionale del Mugello,1,0.83,0.225,0.156,1.00,False
8,jeddah,Jeddah Corniche Circuit,5,2.06,0.150,0.142,2.09,False
9,yas_marina,Yas Marina Circuit,8,2.11,0.143,0.138,2.12,False


In [4]:
# @title Choose the race and the field
BASE_YEAR = 2025  # @param {type:"integer"}
BASE_ROUND = 24  # @param {type:"integer"}
CIRCUIT = ""  # @param {type:"string"}
DRIVER_SWAPS = ""  # @param {type:"string"}
CAR_CHANGES = ""  # @param {type:"string"}
DNFS = True  # @param {type:"boolean"}

# Field: the real field of BASE_YEAR round BASE_ROUND, then your changes.
#   CIRCUIT       blank = that race's circuit, or a CircuitId from the table above (e.g. monaco)
#   DRIVER_SWAPS  OLD=NEW or OLD=NEW@SEASON, ';'-separated. The new driver takes the old one's car.
#                 e.g.  TSU=HAM@2019; LAW=VET@2018      (no @SEASON = their latest season)
#   CAR_CHANGES   DRIVER=TEAM or DRIVER=TEAM@SEASON, e.g.  VER=mclaren; HAM=mercedes@2020

field = sim.field_from_race(results, BASE_YEAR, BASE_ROUND)
if DRIVER_SWAPS.strip():
    field = sim.apply_driver_swaps(field, results, post, DRIVER_SWAPS)
if CAR_CHANGES.strip():
    field = sim.apply_car_changes(field, results, CAR_CHANGES)

base = cal[(cal["Year"] == BASE_YEAR) & (cal["Round"] == BASE_ROUND)].iloc[0]
circuit = CIRCUIT.strip() or base["CircuitId"]
if circuit not in set(cal["CircuitId"]):
    raise KeyError(f"Unknown circuit '{circuit}'. Options: {sorted(cal['CircuitId'].unique())}")
row = thresholds[thresholds["CircuitId"] == circuit]
threshold = float(row["Threshold"].iloc[0]) if len(row) else float(thresholds["Threshold"].median())
entries = sim.describe_field(field, results, post, circuit)

print(f"Circuit: {cal.loc[cal['CircuitId'] == circuit, 'CircuitName'].iloc[0]} | "
      f"overtaking threshold {threshold:.2f}% of a lap"
      + ("" if len(row) else " (no dry races here: median used)"))
per_team = entries.groupby(["TeamId", "CarYear"]).size()
if (per_team != 2).any():
    print("Note: some teams don't have exactly two cars:", per_team[per_team != 2].to_dict())
entries.drop(columns=["_skill_cell", "_car_cells", "DriverId", "TeamId"])

Circuit: Yas Marina Circuit | overtaking threshold 0.14% of a lap


,Driver,FullName,SkillYear,Team,CarYear,CarValue,Swapped
0,VER,Max Verstappen,2025,Red Bull,2025,at this circuit,False
1,PIA,Oscar Piastri,2025,McLaren,2025,at this circuit,False
2,NOR,Lando Norris,2025,McLaren,2025,at this circuit,False
3,LEC,Charles Leclerc,2025,Ferrari,2025,at this circuit,False
4,RUS,George Russell,2025,Mercedes,2025,at this circuit,False
5,ALO,Fernando Alonso,2025,Aston Martin,2025,at this circuit,False
6,OCO,Esteban Ocon,2025,Haas F1 Team,2025,at this circuit,False
7,HAM,Lewis Hamilton,2025,Ferrari,2025,at this circuit,False
8,HUL,Nico Hülkenberg,2025,Sauber,2025,at this circuit,False
9,STR,Lance Stroll,2025,Aston Martin,2025,at this circuit,False


In [5]:
# @title Simulate
N_SIMS = 5000  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}

res = sim.simulate(entries, post, circuit, threshold, dnf if DNFS else None,
                   n_sims=N_SIMS, seed=SEED)
summary = res.summary()
if summary["Swapped"].any():
    print("Swapped entries (driver in a car they didn't race that season): "
          + ", ".join(summary.loc[summary["Swapped"], "Driver"])
          + ". Treat their results as more uncertain than the intervals show.")
summary

,Driver,FullName,SkillYear,Team,CarYear,Swapped,Win%,Podium%,Points%,DNF%,Pole%,AvgGrid,AvgFinish,ExpPoints,ExpPointsLow90,ExpPointsHigh90
0,NOR,Lando Norris,2025,McLaren,2025,False,37.76,79.56,89.18,10.30,22.48,3.79,2.12,17.51,0.0,25.0
1,VER,Max Verstappen,2025,Red Bull,2025,False,30.46,77.52,92.14,7.34,18.74,4.24,2.39,17.01,0.0,25.0
2,PIA,Oscar Piastri,2025,McLaren,2025,False,22.28,70.48,87.50,11.84,16.52,4.44,2.63,15.34,0.0,25.0
3,RUS,George Russell,2025,Mercedes,2025,False,3.14,23.86,85.08,8.28,5.74,7.39,5.37,9.47,0.0,18.0
4,LEC,Charles Leclerc,2025,Ferrari,2025,False,1.48,10.52,72.20,12.82,3.90,8.87,7.11,6.30,0.0,15.0
5,ALO,Fernando Alonso,2025,Aston Martin,2025,False,0.78,8.10,68.88,13.44,3.58,9.24,7.59,5.55,0.0,15.0
6,ANT,Andrea Kimi Antonelli,2025,Mercedes,2025,False,0.48,5.64,63.30,9.64,3.02,10.01,8.58,4.60,0.0,15.0
7,HAM,Lewis Hamilton,2025,Ferrari,2025,False,0.40,3.30,50.40,13.34,2.68,10.86,9.83,3.19,0.0,12.0
8,BEA,Oliver Bearman,2025,Haas F1 Team,2025,False,0.30,2.90,49.74,9.38,2.56,11.18,10.08,3.05,0.0,12.0
9,SAI,Carlos Sainz,2025,Williams,2025,False,0.32,2.38,41.86,16.32,1.94,11.68,10.62,2.44,0.0,10.0


In [6]:
# @title Win, podium and points chances
long = summary.melt(id_vars=["Driver", "Team"], value_vars=["Win%", "Podium%", "Points%"],
                    var_name="Outcome", value_name="Percent")
fig = px.bar(long, x="Driver", y="Percent", color="Outcome", barmode="group",
             hover_data=["Team"], category_orders={"Driver": summary["Driver"].tolist()},
             title=f"Simulated chances over {N_SIMS:,} races")
fig.show()

In [7]:
# @title Finishing position probabilities
probs = res.position_probabilities().loc[summary["Driver"]]
fig = px.imshow(probs, text_auto=".0f", aspect="auto", color_continuous_scale="Blues",
                labels=dict(x="Finishing position", y="Driver", color="% of races"),
                title="Chance of each finishing position (%)")
fig.update_layout(height=700)
fig.show()

In [8]:
# @title Expected points per team
res.team_points()

,Team,Cars,Drivers,ExpPoints,Low90,High90
0,McLaren 2025,2,"PIA, NOR",32.85,15.0,43.0
1,Red Bull 2025,2,"VER, TSU",19.24,1.0,33.0
2,Mercedes 2025,2,"RUS, ANT",14.07,0.0,27.0
3,Ferrari 2025,2,"LEC, HAM",9.49,0.0,23.0
4,Aston Martin 2025,2,"ALO, STR",7.28,0.0,19.0
5,Haas F1 Team 2025,2,"OCO, BEA",5.03,0.0,16.0
6,Williams 2025,2,"SAI, ALB",4.35,0.0,15.0
7,Sauber 2025,2,"HUL, BOR",4.16,0.0,15.0
8,RB F1 Team 2025,2,"HAD, LAW",3.48,0.0,14.0
9,Alpine F1 Team 2025,2,"GAS, COL",1.04,0.0,8.0
